# Revolut FAQ RAG Chatbot

A minimal single-turn RAG chatbot over Revolut help articles.

**Stack:** `openai` for embeddings + chat, `numpy` for similarity search, `json` for loading. No frameworks, no vector DB — everything in memory.

In [1]:
%pip install -q openai numpy

/Users/veniamin/Projects/chatbot-evals-ai/.venv/bin/python3: No module named pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import asyncio
import os
from pathlib import Path
import numpy as np
import pandas as pd
from openai import OpenAI, AsyncOpenAI
from tqdm.asyncio import tqdm
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Import config - single source of truth
import sys
sys.path.insert(0, str(Path(os.getcwd()).parent.parent))
from src.config import *

print(f"Using EMBED_MODEL: {EMBED_MODEL}")
print(f"Using CHAT_MODEL: {CHAT_MODEL}")
print(f"Using TOP_K: {TOP_K}")

# Validate articles path exists
assert ARTICLES_PATH.exists(), f"Articles file not found: {ARTICLES_PATH}"
print(f"Articles path: {ARTICLES_PATH}")

# Initialize clients
client = OpenAI(api_key=OPENAI_API_KEY)
async_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

# Load system prompt from file
with open(RAG_SYSTEM_PROMPT_PATH, 'r') as f:
    SYSTEM_PROMPT = f.read().strip()
print(f"Loaded system prompt from {RAG_SYSTEM_PROMPT_PATH}")

Using EMBED_MODEL: text-embedding-3-small
Using CHAT_MODEL: gpt-3.5-turbo
Using TOP_K: 4
Articles path: /Users/veniamin/Projects/chatbot-evals-ai/data/revolut_help_articles.jsonl
Loaded system prompt from /Users/veniamin/Projects/chatbot-evals-ai/prompts/rag_system.txt


## 1. Load articles

In [3]:
articles = []
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        articles.append(json.loads(line))

print(f"Loaded {len(articles)} articles from {ARTICLES_PATH}")
print("Example:", articles[0]["title"])

Loaded 786 articles from /Users/veniamin/Projects/chatbot-evals-ai/data/revolut_help_articles.jsonl
Example: How can I see my cashflow analytics?


## 2. Embed all articles

We embed `title + content_text` so the title contributes to retrieval. Batched to keep things fast.

In [4]:
def article_to_text(a):
    return f"{a['title']}\n\n{a['content_text']}"

def embed_texts(texts, batch_size=100):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    return np.array(out, dtype=np.float32)

texts = [article_to_text(a) for a in articles]
embeddings = embed_texts(texts)

# L2-normalize once so cosine similarity is just a dot product
embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)

print("Embeddings shape:", embeddings.shape)

Embeddings shape: (786, 1536)


## 3. Retrieval

In [5]:
def retrieve(query, k=TOP_K):
    q_emb = client.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    q_vec = np.array(q_emb, dtype=np.float32)
    q_vec = q_vec / np.linalg.norm(q_vec)

    scores = embeddings @ q_vec
    top_idx = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), articles[i]) for i in top_idx]

## 4. Single-turn chat

In [6]:
# SYSTEM_PROMPT is now loaded from prompts/rag_system.txt via config

def format_context(hits):
    parts = []
    for rank, (idx, score, art) in enumerate(hits, start=1):
        parts.append(
            f"[Article {rank}] {art['title']}\n{art['content_text']}"
        )
    return "\n\n---\n\n".join(parts)

def ask(question, k=TOP_K):
    """Sync ask function - thin wrapper over async_answer_with_context."""
    answer, context, hits = asyncio.run(async_answer_with_context(question, k=k))
    return answer, hits

async def async_answer_with_context(query, k=TOP_K):
    """
    Async RAG query that returns answer, extracted context, and hits.
    Uses AsyncOpenAI for both embedding and chat calls.
    """
    # Async embedding
    q_emb_resp = await async_client.embeddings.create(
        model=EMBED_MODEL,
        input=[query]
    )
    q_emb = q_emb_resp.data[0].embedding
    q_vec = np.array(q_emb, dtype=np.float32)
    q_vec = q_vec / np.linalg.norm(q_vec)
    
    # Retrieval
    scores = embeddings @ q_vec
    top_idx = np.argsort(-scores)[:k]
    hits = [(int(i), float(scores[i]), articles[i]) for i in top_idx]
    
    # Format context
    context = format_context(hits)
    
    # Async chat completion
    user_msg = (
        f"Help articles:\n\n{context}\n\n"
        f"Question: {query}"
    )
    
    resp = await async_client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg},
        ],
        temperature=0.2,
    )
    answer = resp.choices[0].message.content
    
    return answer, context, hits

## 5. Try it

In [7]:
question = "как открыть аккаунт в монзо"

# Test async_answer_with_context
answer, context, hits = await async_answer_with_context(question)

print("Q:", question)
print("\nA:", answer)
print("\nContext (first 200 chars):", context[:200] + "...")
print("\nRetrieved articles:")
for rank, (idx, score, art) in enumerate(hits, start=1):
    print(f"  {rank}. [{score:.3f}] {art['title']}")

Q: как открыть аккаунт в монзо

A: I'm sorry, I don't have information on how to open an account with Monzo.

Context (first 200 chars): [Article 1] Open a Revolut – Kids & Teens account
## Create an account for your kids or teens
In the main Revolut app: 
- Go to 'Home' on the bottom menu
- Below your balance, tap Accounts
- Tap 'Add ...

Retrieved articles:
  1. [0.369] Open a Revolut – Kids & Teens account
  2. [0.357] Duplicate account
  3. [0.337] Open an investment account
  4. [0.330] Change or verify email address


## 6. Evaluate Synthetic Dataset

Evaluate the RAG assistant on the synthetic dataset of 1500 queries.

In [ ]:
# Load synthetic dataset
DATASET_PATH = DATA_DIR / "synthetic_revolut_queries.csv"
df_queries = pd.read_csv(DATASET_PATH)
print(f"Loaded {len(df_queries)} synthetic queries")
print(f"Columns: {list(df_queries.columns)}")
df_queries.head()

In [ ]:
# Evaluation settings
RAG_OUTPUT_PATH = DATA_DIR / "synthetic_revolut_rag_outputs.csv"
ROW_KEY = ["persona", "scenario", "modifier", "query"]
MAX_EVAL_ROWS = None  # FULL DATASET - 1500 queries

# Resume logic: load existing results if any
completed_keys = set()
if RAG_OUTPUT_PATH.exists():
    df_existing = pd.read_csv(RAG_OUTPUT_PATH)
    completed_keys = set(zip(df_existing["persona"], df_existing["scenario"], df_existing["modifier"], df_existing["query"]))
    print(f"RAG resume: {len(completed_keys)} done, {len(df_queries) - len(completed_keys)} missing")
else:
    print(f"Starting fresh RAG evaluation")

# Filter to rows we need to process
if MAX_EVAL_ROWS:
    df_todo = df_queries.head(MAX_EVAL_ROWS)
else:
    df_todo = df_queries[~df_queries.apply(lambda r: (r["persona"], r["scenario"], r["modifier"], r["query"]) in completed_keys, axis=1)]

print(f"Will process {len(df_todo)} queries (FULL DATASET)")

In [ ]:
# Cost gate: estimate API calls (2 per query: embedding + chat)
estimated_calls = len(df_todo) * 2
print(f"Estimated API calls: {estimated_calls}")
if estimated_calls > 2000:
    print("⚠️  This will make >2000 API calls.")
    # Uncomment below to enforce confirmation
    # response = input("Continue? (yes/no): ")
    # if response.lower() != "yes":
    #     raise SystemExit("Aborted by user")

In [ ]:
# Async evaluation loop with checkpointing
async def evaluate_row(row):
    """Evaluate a single query through RAG."""
    answer, context, hits = await async_answer_with_context(row["query"])
    return {
        "persona": row["persona"],
        "scenario": row["scenario"],
        "modifier": row["modifier"],
        "query": row["query"],
        "answer": answer,
        "extracted_context": context,
        "retrieved_articles": " | ".join([art["title"] for _, _, art in hits])
    }

async def run_evaluation(df, save_every=SAVE_EVERY):
    """Run evaluation - save once at end for full dataset."""
    results = []
    semaphore = asyncio.Semaphore(RAG_CONCURRENCY)
    
    async def process_with_semaphore(row):
        async with semaphore:
            return await evaluate_row(row)
    
    # Create tasks
    tasks = [process_with_semaphore(row) for _, row in df.iterrows()]
    
    # Process with progress tracking
    for i, future in enumerate(tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="RAG evaluation")):
        result = await future
        results.append(result)
        
        # Progress checkpoint (no saving during full run)
        if (i + 1) % save_every == 0:
            print(f"Processed: {i + 1}/{len(tasks)} queries")
    
    # Final save
    final_df = pd.DataFrame(results)
    final_df.to_csv(RAG_OUTPUT_PATH, index=False)
    print(f"Saved {len(final_df)} results to {RAG_OUTPUT_PATH}")
    
    return final_df

# Run evaluation
df_results = await run_evaluation(df_todo)
print(f"Evaluation complete: {len(df_results)} results")

In [ ]:
# Gate G4 - Verify RAG outputs
print("="*60)
print("Gate G4 - RAG Output Validation")
print("="*60)

df_final = pd.read_csv(RAG_OUTPUT_PATH)

# Row count
print(f"Row count: {len(df_final)} (expected: 1500)")

# Columns
expected_cols = ["persona", "scenario", "modifier", "query", "answer", "extracted_context", "retrieved_articles"]
print(f"Columns correct: {list(df_final.columns) == expected_cols}")

# Null checks
null_answers = df_final["answer"].isnull().sum()
null_context = df_final["extracted_context"].isnull().sum()
print(f"Null answers: {null_answers}")
print(f"Null context: {null_context}")

# Retrieved articles
empty_retrieved = (df_final["retrieved_articles"].str.len() == 0).sum()
print(f"Empty retrieved_articles: {empty_retrieved} ({empty_retrieved/len(df_final)*100:.1f}%)")

# Sample outputs
print(f"\nSample rows:")
for i, row in df_final.sample(3).iterrows():
    print(f"\n[{row['modifier'][:15]:15}] Query: {row['query'][:60]}...")
    print(f"Answer: {row['answer'][:80]}...")
    print(f"Retrieved: {row['retrieved_articles'][:100]}...")

# Final verdict
g4_pass = (len(df_final) == 1500 and 
           list(df_final.columns) == expected_cols and
           null_answers == 0 and 
           empty_retrieved / len(df_final) <= 0.01)

print(f"\n{'✅ Gate G4 PASSED' if g4_pass else '❌ Gate G4 FAILED'}")

## Milestone 0 Verification

Verify .gitignore excludes .env but not data CSVs:

In [8]:
# Check .gitignore status
gitignore_path = Path(os.getcwd()).parent.parent / ".gitignore"
with open(gitignore_path) as f:
    gitignore = f.read()
print(".env in .gitignore:", ".env" in gitignore)
print("data/*.csv in .gitignore:", "data/*.csv" in gitignore)
print("First 20 lines of .gitignore:")
print('\n'.join(gitignore.split('\n')[:20]))

.env in .gitignore: True
data/*.csv in .gitignore: False
First 20 lines of .gitignore:
node_modules
.next
out
dist
.env*
.DS_Store
*.log
*.sqlite
coverage

# local archives
*.zip
*.tar.gz

